In [1]:
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.transforms as T
import torch
import numpy as np
import os.path as osp
from torch_geometric.nn import GCNConv, SAGEConv
import torch.nn.functional as F
from deeprobust.graph.utils import *
import torch_sparse
import logging
from utils import scale_feats

torch.cuda.set_device(2)  # 指第几块GPU
root_path = osp.expanduser('/home/songsh/GCL/datasets_pyg')

dataname = 'ogbn-arxiv'
dataset = PygNodePropPredDataset(root=osp.join(root_path, 'OGB'), name=dataname,
                                     transform=T.ToSparseTensor())
data = dataset[0]

# process features
feat = data.x 
data.x = scale_feats(feat)

data.adj_t = data.adj_t.to_symmetric()
device = torch.device('cuda')
evaluator = Evaluator(name='ogbn-arxiv')

data = data.to(device)
split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)
val_idx = split_idx['valid'].to(device)
test_idx = split_idx['test'].to(device)
ptb_rate = 0.5

# there is only one graph in Node Property Prediction datasets
# 在Node Property Prediction数据集里只有一个图
# 获取划分的标签
# train_label = dataset.labels[split_idx['train']]
# valid_label = dataset.labels[split_idx['valid']]
# test_label = dataset.labels[split_idx['test']]

/home/songsh/anaconda3/anaconda3_install/envs/ProGCL/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using backend: pytorch


In [4]:
# test的数量少，所以根据影响分布的程度，这里倾向于攻击test-test和test-train
def heuristic_attack(perturbed_adj, n_perturbations, idx_train, idx_test, labels):
    degree = torch.tensor(perturbed_adj.sum(axis=1)).to('cuda').reshape(-1)
    canditate_train_idx = idx_train[degree[idx_train] < 10]
    candidate_test_idx = idx_test[degree[idx_test] < 10]
    
    canditate_train_idx = canditate_train_idx.to('cpu').numpy()
    candidate_test_idx = candidate_test_idx.to('cpu').numpy()
    cnt = 0
    n_test_train = int(n_perturbations / 3) 
    n_test_test = n_perturbations - n_test_train  
    while cnt < n_test_test:
        n_remaining = n_test_test - cnt
        # sample random pairs
        candidate_edges = np.array([np.random.choice(candidate_test_idx, n_remaining),
                                    np.random.choice(candidate_test_idx, n_remaining)]).T

        # filter out existing edges, and pairs with the different labels
        candidate_edges = set([(u, v) for u, v in candidate_edges if labels[u] != labels[v]
                                    and perturbed_adj[u, v] == 0 and perturbed_adj[v, u] == 0])
        candidate_edges = np.array(list(candidate_edges))

        # if none is found, try again
        if len(candidate_edges) == 0:
            continue

        # add all found edges to your modified adjacency matrix
        perturbed_adj[candidate_edges[:, 0], candidate_edges[:, 1]] = 1
        perturbed_adj[candidate_edges[:, 1], candidate_edges[:, 0]] = 1
        cnt += candidate_edges.shape[0]
        # print(cnt)
    cnt = 0
    while cnt < n_test_train:
        n_remaining = n_test_train - cnt
        # sample random pairs
        candidate_edges = np.array([np.random.choice(candidate_test_idx, n_remaining),
                                    np.random.choice(canditate_train_idx, n_remaining)]).T

        # filter out existing edges, and pairs with the different labels
        candidate_edges = set([(u, v) for u, v in candidate_edges if labels[u] != labels[v]
                                    and perturbed_adj[u, v] == 0 and perturbed_adj[v, u] == 0])
        candidate_edges = np.array(list(candidate_edges))

        # if none is found, try again
        if len(candidate_edges) == 0:
            continue

        # add all found edges to your modified adjacency matrix
        perturbed_adj[candidate_edges[:, 0], candidate_edges[:, 1]] = 1
        perturbed_adj[candidate_edges[:, 1], candidate_edges[:, 0]] = 1
        cnt += candidate_edges.shape[0]


In [5]:
adj = torch_sparse.tensor.SparseTensor.to_torch_sparse_coo_tensor(data.adj_t)
adj = to_scipy(adj)
n_perturbations = int(ptb_rate * (adj.sum()//2))
print('num ptb: ' +  str(n_perturbations))
labels = data.y
import time
start = time.time()
# model = DICE()
# model.attack(adj, labels, n_perturbations)
# adj = model.modified_adj
# DICE only attack the testing set

heuristic_attack(adj, n_perturbations, train_idx, test_idx, labels)
print(time.time() - start)
perturbed_adj = torch_sparse.tensor.from_scipy(adj)
data.adj_t = perturbed_adj.to(device)

num ptb: 578899


/home/songsh/anaconda3/anaconda3_install/envs/ProGCL/lib/python3.7/site-packages/scipy/sparse/_index.py:125: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


100.18189024925232


In [ ]:
def get_logger(filename, verbosity=1, name=None):
    level_dict = {0: logging.DEBUG, 1: logging.INFO, 2: logging.WARNING}
    formatter = logging.Formatter(
        "[%(asctime)s][%(filename)s][line:%(lineno)d][%(levelname)s] %(message)s"
    )
    logger = logging.getLogger(name)
    logger.setLevel(level_dict[verbosity])

    fh = logging.FileHandler(filename, "w")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    sh = logging.StreamHandler()
    sh.setFormatter(formatter)
    logger.addHandler(sh)

    return logger

logger = get_logger('./log/heuristic_attack_%s_%s.log' % ('ogb', str(ptb_rate)))

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers,
                 dropout):
        super(GCN, self).__init__()

        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels, cached=True))
        self.bns = torch.nn.ModuleList()
        self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(
                GCNConv(hidden_channels, hidden_channels, cached=True))
            self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels, cached=True))

        self.dropout = dropout

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adj_t):
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, adj_t)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, adj_t)
        return x.log_softmax(dim=-1)
    
    
def train(model, data, train_idx, optimizer):
    model.train()

    optimizer.zero_grad()
    out = model(data.x, data.adj_t)[train_idx]
    loss = F.nll_loss(out, data.y.squeeze(1)[train_idx])
    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def test(model, data, split_idx, evaluator):
    model.eval()

    out = model(data.x, data.adj_t)
    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': data.y[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': data.y[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': data.y[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    return train_acc, valid_acc, test_acc

In [6]:
gcn = GCN(in_channels=data.num_features,
          hidden_channels=256,
          out_channels=dataset.num_classes,
          num_layers=3,
          dropout=0.5).to(device)
acc = []
for run in range(1):
    gcn.reset_parameters()
    optimizer = torch.optim.Adam(gcn.parameters(), lr=0.01)
    for epoch in range(1, 501):
        loss = train(gcn, data, train_idx, optimizer)
        result = test(gcn, data, split_idx, evaluator)
        if (epoch % 10) == 0:
            train_acc, valid_acc, test_acc = result
            acc.append(test_acc)
            logger.info(f'Run: {run + 1:02d}, '
                  f'Epoch: {epoch:02d}, '
                  f'Loss: {loss:.4f}, '
                  f'Train: {100 * train_acc:.2f}%, '
                  f'Valid: {100 * valid_acc:.2f}% '
                  f'Test: {100 * test_acc:.2f}%')

INFO:root:Run: 01, Epoch: 10, Loss: 1.6118, Train: 14.78%, Valid: 13.88% Test: 8.55%
[2023-12-05 22:44:47,818][3603473944.py][line:16][INFO] Run: 01, Epoch: 10, Loss: 1.6118, Train: 14.78%, Valid: 13.88% Test: 8.55%
INFO:root:Run: 01, Epoch: 20, Loss: 1.3654, Train: 48.13%, Valid: 55.78% Test: 34.50%
[2023-12-05 22:44:49,108][3603473944.py][line:16][INFO] Run: 01, Epoch: 20, Loss: 1.3654, Train: 48.13%, Valid: 55.78% Test: 34.50%
INFO:root:Run: 01, Epoch: 30, Loss: 1.2511, Train: 60.72%, Valid: 64.72% Test: 44.91%
[2023-12-05 22:44:50,399][3603473944.py][line:16][INFO] Run: 01, Epoch: 30, Loss: 1.2511, Train: 60.72%, Valid: 64.72% Test: 44.91%
INFO:root:Run: 01, Epoch: 40, Loss: 1.1882, Train: 65.41%, Valid: 67.90% Test: 47.69%
[2023-12-05 22:44:51,691][3603473944.py][line:16][INFO] Run: 01, Epoch: 40, Loss: 1.1882, Train: 65.41%, Valid: 67.90% Test: 47.69%
INFO:root:Run: 01, Epoch: 50, Loss: 1.1386, Train: 67.46%, Valid: 68.92% Test: 48.38%
[2023-12-05 22:44:52,982][3603473944.py][lin

In [7]:
#save
import scipy
from datasets_dgl.utils import sparse_mx_to_torch_sparse_tensor
save_path = '/home/songsh/SPMGAE/datasets_dgl/all_data_attack/heuristic/'

perturbed_adj = sparse_mx_to_torch_sparse_tensor(adj)
features  = data.x.cpu()
labels    = data.y.squeeze().cpu()
train_idx = train_idx.cpu()
val_idx   = val_idx.cpu()
test_idx  = test_idx.cpu()


torch.save(perturbed_adj,   save_path +  '%s/heuristic_%s_%s.pt' % (dataname,  dataname, ptb_rate))

if ptb_rate == 0: #保存一次label和features即可
    scipy.sparse.save_npz(save_path + '%s/%s_features' % (dataname,  dataname), scipy.sparse.csr_matrix(np.array(features)))
    np.save(save_path + '%s/%s_labels' % (dataname, dataname),   labels)

np.save( save_path + '%s/heuristic_%s_%s_idx_train' % (dataname, dataname, ptb_rate),   train_idx)
np.save( save_path + '%s/heuristic_%s_%s_idx_val'   % (dataname, dataname, ptb_rate),     val_idx)
np.save(save_path + '%s/heuristic_%s_%s_idx_test'   % (dataname, dataname, ptb_rate),    test_idx)

In [8]:
from   datasets_dgl.data_dgl  import load_attack_data
from     easydict        import EasyDict

# dataset_name = dataset_name.split('-')[1]
DATASET = EasyDict()
DATASET.ATTACK = {
    "data"    : 'Attack-ogbn-arxiv',
    "attack"  : 'heuristic',
    "ptb_rate":  str(ptb_rate)
}
# now just attack use
dataset  = load_attack_data(DATASET['ATTACK'])
graph = dataset.graph
graph

[I] Loading dataset Attack-ogbn-arxiv...
train_mask, test_mask, val_mask sizes : tensor(90941) tensor(29799) tensor(48603)
[I] Finished loading.
[I] Data load time: 4.7868s


Graph(num_nodes=169343, num_edges=3642541,
      ndata_schemes={'feat': Scheme(shape=(128,), dtype=torch.float32), 'label': Scheme(shape=(1,), dtype=torch.int64), 'train_mask': Scheme(shape=(), dtype=torch.bool), 'val_mask': Scheme(shape=(), dtype=torch.bool), 'test_mask': Scheme(shape=(), dtype=torch.bool)}
      edata_schemes={})